## Where each model is used in this notebook

| Role | Model | Where in the code |
|---|---|---|
| **Bi-encoder** (stage 1, dense retrieval) | `BAAI/bge-small-en-v1.5` | Section 2: `bi = SentenceTransformer(...)`, chunk vectors stored in a FAISS inner-product index; used by `retrieve_top10()` in section 3 |
| **Cross-encoder** (stage 2, re-ranking) | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Section 2: `ce = CrossEncoder(...)`; used by `rerank()` in section 3 |
| **LLM** (answer generation) | Gemini `gemini-3.5-flash-lite` | Section 3: `ask_llm()` |

**EmbeddingGemma 2 is not used in this notebook.** It is compared against bge-small in `rag_retrieval_25q.ipynb` (25 questions) and `rag_experiments.ipynb` (8 questions).

Data: the SRD PDF is in `data/SRD_CC_v5.1.pdf`. The chunks retrieved for every question (bi-encoder order, cross-encoder order, scores and chunk text) are saved in `outputs_25q/retrieved_chunks.md` and `outputs_25q/retrieval.json`; the first 8 questions there are the ones in this notebook.

Note: the paths in the code cells are the cluster paths this notebook was run with; point `PDF` and `OUT` in the first code cell to your own copies.

# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking

**Corpus:** D&D 5e System Reference Document 5.1 (403-page, two-column PDF, CC-BY).
**Bi-encoder:** `BAAI/bge-small-en-v1.5` (dense, cosine / inner product on normalised vectors, FAISS).
**Cross-encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`.
**LLM:** Gemini (`gemini-3.5-flash-lite`, temperature 0) through the Google GenAI SDK. The API key is read from the `GEMINI_API_KEY` environment variable and is never stored in this notebook.

Both pipelines use the *same* chunks, the *same* top-10 candidates and the *same* prompt. The only difference is which 3 chunks reach the LLM.

In [1]:
import os, re, time, json, textwrap
import numpy as np, pandas as pd, faiss, pymupdf, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

PDF = "/home2/akshat.pandey/rag/srd.pdf"
OUT = "/home2/akshat.pandey/rag/outputs"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_MODEL = "gemini-3.5-flash-lite"   # free-tier key: gemini-3.8-flash hit its daily request cap, so one lighter model is used for ALL answers
print("device:", DEVICE, "| gpu:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-")

device: cuda | gpu: NVIDIA GeForce RTX 2080 Ti


## 1. Load the PDF and chunk it

The SRD is laid out in two columns. Reading a page in plain order interleaves the columns and produces garbled sentences,
so each page is cut into a left and a right half and each half is read separately. A 40 pt strip at the top and bottom
(running header / footer) is dropped. The text is then split into overlapping windows of 200 words (40-word overlap) so a
rule that straddles a boundary still appears whole in at least one chunk.

In [2]:
CHUNK_WORDS, OVERLAP = 200, 40

doc = pymupdf.open(PDF)
words, word_page = [], []
for pno, page in enumerate(doc, 1):
    r = page.rect
    top, bot, mid = r.y0 + 40, r.y1 - 40, (r.x0 + r.x1) / 2
    for col in (pymupdf.Rect(r.x0, top, mid, bot), pymupdf.Rect(mid, top, r.x1, bot)):
        for w in page.get_text("text", clip=col).split():
            words.append(w); word_page.append(pno)

chunks = [{"id": i, "page": word_page[s], "text": " ".join(words[s:s + CHUNK_WORDS])}
          for i, s in enumerate(range(0, len(words), CHUNK_WORDS - OVERLAP))]
print(f"{len(doc)} pages, {len(words)} words -> {len(chunks)} chunks")
print("\nSample chunk:\n", textwrap.fill(chunks[300]["text"], 110))

403 pages, 240639 words -> 1504 chunks

Sample chunk:
 clear on the jump costs a foot of movement. In some circumstances, your GM might allow you to make a Strength
(Athletics) check to jump higher than you normally can. You can extend your arms half your height above
yourself during the jump. Thus, you can reach above you a distance equal to the height of the jump plus 1½
times your height. The Environment By its nature, adventuring involves delving into places that are dark,
dangerous, and full of mysteries to be explored. The rules in this section cover some of the most important
ways in which adventurers interact with the environment in such places. Falling A fall from a great height is
one of the most common hazards facing an adventurer. At the end of a fall, a creature takes 1d6 bludgeoning
damage for every 10 feet it fell, to a maximum of 20d6. The creature lands prone, unless it avoids taking
damage from the fall. Suffocating A creature can hold its breath for a number of minut

## 2. Models and vector index (Bi-Encoder)

Chunk embeddings are L2-normalised, so the inner product equals cosine similarity. BGE expects an instruction prefix on
*queries* (not on documents).

In [3]:
bi = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

t0 = time.perf_counter()
emb = bi.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)
print(f"embedded {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s, dim={emb.shape[1]}")
index = faiss.IndexFlatIP(emb.shape[1])   # exact inner-product search
index.add(emb)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

embedded 1504 chunks in 5.2s, dim=384


## 3. The two pipelines

* `retrieve_top10` : Stage 1, bi-encoder + FAISS (this IS Pipeline 1's retrieval).
* `rerank`         : Stage 2, cross-encoder scores all 10 `(query, chunk)` pairs jointly and re-sorts.
* `build_prompt` / `ask_llm` : identical for both pipelines.

In [4]:
def retrieve_top10(query, k=10):
    q = bi.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return [{"id": int(i), "bi_score": float(s)} for i, s in zip(ids[0], scores[0])]

def rerank(query, cands):
    pairs = [(query, chunks[c["id"]]["text"]) for c in cands]
    ce_scores = ce.predict(pairs, batch_size=10)
    out = [dict(c, ce_score=float(s)) for c, s in zip(cands, ce_scores)]
    return sorted(out, key=lambda c: -c["ce_score"])

PROMPT = """You are a rules assistant for Dungeons & Dragons 5th edition.
Answer the question using ONLY the context passages below. If the passages do not contain the answer, say
"The provided context does not answer this." Be concise (at most 4 sentences) and state any exception or restriction that applies.

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(query, top3):
    ctx = "\n\n".join(f"[Passage {n+1}, page {chunks[c['id']]['page']}] {chunks[c['id']]['text']}"
                      for n, c in enumerate(top3))
    return PROMPT.format(context=ctx, question=query)

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

import hashlib
CACHE_FILE = f"{OUT}/llm_cache.json"
llm_cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def ask_llm(prompt, retries=5):
    """One Gemini call -> (answer, seconds taken by the successful call).
    Free-tier keys are rate limited: pause between calls, back off on a per-minute 429, and give up at once on a
    per-day 429 (retrying cannot help). Answers are cached on disk by prompt hash so a re-run costs no quota."""
    key = hashlib.sha1((LLM_MODEL + prompt).encode()).hexdigest()
    if key in llm_cache:
        return llm_cache[key]["answer"], llm_cache[key]["secs"]
    for attempt in range(retries):
        try:
            time.sleep(6)
            t = time.perf_counter()
            r = client.models.generate_content(model=LLM_MODEL, contents=prompt,
                                               config=types.GenerateContentConfig(temperature=0))
            secs = time.perf_counter() - t
            llm_cache[key] = {"answer": (r.text or "").strip(), "secs": secs}
            json.dump(llm_cache, open(CACHE_FILE, "w"))
            return llm_cache[key]["answer"], secs
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg:
                return f"[LLM ERROR: daily quota exhausted]", float("nan")
            if any(code in msg for code in ("429", "503", "500")) and attempt < retries - 1:
                time.sleep(15 * 2 ** attempt)
            else:
                return f"[LLM ERROR: {msg[:200]}]", float("nan")

## 4. Failure-mode test queries

For each query I picked, by reading the SRD, a short phrase that appears only in the passage that actually answers it
(`gold`). A chunk counts as "gold" if it contains that phrase (a regex, so apostrophes match). The "gold rank" is the
position of the best-ranked such chunk. The queries cover the categories the assignment asks for, plus two questions
(the last two) where the answer chunk is not among the ten candidates at all, so re-ranking cannot help. I keep them in on purpose.
Each phrase was checked by printing the chunk it matches (see `outputs_25q/gold_phrase_check.txt`).

In [5]:
TESTS = [
 # (category, query, gold regex)
 ("Keyword trap",          "What breaks my focus on a spell?",                              r"incapacitated or killed"),
 ("Negation / constraint", "Do I need to see a target to attack it?",                       r"target that you can.t see"),
 ("Exception / corner",    "Can I punish an unseen enemy that runs away from me?",          r"hostile creature that you can see moves out of your reach"),
 ("Negation / constraint", "Can I cast a spell while a silence effect is on me?",           r"can.t cast a spell with a verbal component"),
 ("Exception / corner",    "Can I shove a creature much bigger than me?",                   r"push it away from you"),
 ("Keyword trap (synonym)","How many magic items can one creature bond with?",              r"no more than three magic items"),
 ("Counter-example",       "Can I rest twice in a single day to recover everything?",       r"more than one long rest in a 24"),
 ("Counter-example",       "Can a creature that is held by an enemy still walk away?",      r"grappled creature.s speed becomes 0"),
]

def gold_rank(order, gold):
    g = re.compile(gold, re.I)
    for r, c in enumerate(order, 1):
        if g.search(chunks[c["id"]]["text"]):
            return r, c["id"]
    return None, None

## 5. Run both pipelines on every test query

Pipeline 2 receives exactly the same ten candidates as Pipeline 1 (asserted in the code), so any difference in the final
answer is caused only by the re-ranking.

In [6]:
results = []
for cat, q, gold in TESTS:
    # ---- Stage 1 (shared by both pipelines) ----
    t = time.perf_counter(); cands = retrieve_top10(q); t_retr = time.perf_counter() - t
    # ---- Pipeline 1: top-3 straight from the bi-encoder ----
    base_top3 = cands[:3]
    prompt1 = build_prompt(q, base_top3)
    ans1, t_llm1 = ask_llm(prompt1)
    # ---- Pipeline 2: cross-encoder re-sort of the SAME 10, then top-3 ----
    t = time.perf_counter(); reranked = rerank(q, cands); t_rr = time.perf_counter() - t
    assert {c["id"] for c in reranked} == {c["id"] for c in cands}      # same candidate set
    rr_top3 = reranked[:3]
    prompt2 = build_prompt(q, rr_top3)
    ans2, t_llm2 = ask_llm(prompt2)

    g_before, g_id = gold_rank(cands, gold)
    g_after, _ = gold_rank(reranked, gold)
    results.append(dict(category=cat, query=q, gold=gold, gold_chunk=g_id,
                        gold_rank_baseline=g_before, gold_rank_reranked=g_after,
                        base_ids=[c["id"] for c in cands], rerank_ids=[c["id"] for c in reranked],
                        ce_scores={c["id"]: round(c["ce_score"], 3) for c in reranked},
                        answer1=ans1, answer2=ans2,
                        t_retrieval=t_retr, t_rerank=t_rr, t_llm1=t_llm1, t_llm2=t_llm2))

json.dump(results, open(f"{OUT}/results.json", "w"), indent=1)
print("done:", len(results), "queries")

done: 8 queries


## 6. Rank-shift table

`None` means no chunk containing the gold phrase was in the top 10 at all.

In [7]:
summary = pd.DataFrame([{k: r[k] for k in ("category", "query", "gold_chunk", "gold_rank_baseline", "gold_rank_reranked")}
                        for r in results])
summary["in_top3_baseline"] = summary.gold_rank_baseline.apply(lambda x: x is not None and x <= 3)
summary["in_top3_reranked"] = summary.gold_rank_reranked.apply(lambda x: x is not None and x <= 3)
pd.set_option("display.width", 220, "display.max_colwidth", 70)
summary.to_csv(f"{OUT}/rank_shift.csv", index=False)
summary

,category,query,gold_chunk,gold_rank_baseline,gold_rank_reranked,in_top3_baseline,in_top3_reranked
0,Keyword trap,What breaks my focus on a spell?,372.0,5.0,2.0,False,True
1,Negation / constraint,Do I need to see a target to attack it?,337.0,8.0,3.0,False,True
2,Exception / corner,Can I punish an unseen enemy that runs away from me?,340.0,4.0,3.0,False,True
3,Negation / constraint,Can I cast a spell while a silence effect is on me?,369.0,3.0,1.0,True,True
4,Exception / corner,Can I shove a creature much bigger than me?,343.0,3.0,1.0,True,True
5,Keyword trap (synonym),How many magic items can one creature bond with?,781.0,3.0,1.0,True,True
6,Counter-example,Can I rest twice in a single day to recover everything?,NaN,NaN,NaN,False,False
7,Counter-example,Can a creature that is held by an enemy still walk away?,NaN,NaN,NaN,False,False


## 7. Per-query detail: ranks, chunks and the two answers

For each query: the baseline top-10 order, the re-ranked order with cross-encoder scores, the text of the gold chunk,
and Answer 1 (baseline) next to Answer 2 (re-ranked).

In [8]:
for r in results:
    print("=" * 110)
    print(f"[{r['category']}] {r['query']}")
    print(f"gold phrase /{r['gold']}/  -> chunk c{r['gold_chunk']} (page {chunks[r['gold_chunk']]['page']})"
          if r["gold_chunk"] is not None else f"gold phrase /{r['gold']}/ NOT in top 10")
    print(f"gold rank: baseline #{r['gold_rank_baseline']}  ->  re-ranked #{r['gold_rank_reranked']}")
    print("baseline top-10 :", ["c%d" % i for i in r["base_ids"]])
    print("re-ranked top-10:", ["c%d(%.1f)" % (i, r["ce_scores"][i]) for i in r["rerank_ids"]])
    if r["gold_chunk"] is not None:
        print("\nGOLD CHUNK:", textwrap.fill(chunks[r["gold_chunk"]]["text"][:600], 110), "...")
    print("\nANSWER 1 (baseline):\n", textwrap.fill(r["answer1"], 110))
    print("\nANSWER 2 (re-ranked):\n", textwrap.fill(r["answer2"], 110))

[Keyword trap] What breaks my focus on a spell?
gold phrase /incapacitated or killed/  -> chunk c372 (page 102)
gold rank: baseline #5  ->  re-ranked #2
baseline top-10 : ['c334', 'c371', 'c368', 'c222', 'c372', 'c448', 'c483', 'c578', 'c692', 'c574']
re-ranked top-10: ['c334(3.0)', 'c372(-0.3)', 'c371(-0.6)', 'c368(-0.7)', 'c483(-2.0)', 'c222(-3.4)', 'c578(-3.9)', 'c448(-6.8)', 'c692(-8.0)', 'c574(-8.2)']

GOLD CHUNK: on two spells at once. • Taking damage. Whenever you take damage while you are concentrating on a spell, you
must make a Constitution saving throw to maintain your concentration. The DC equals 10 or half the damage you
take, whichever number is higher. If you take damage from multiple sources, such as an arrow and a dragon’s
breath, you make a separate saving throw for each source of damage. • Being incapacitated or killed. You lose
concentration on a spell if you are incapacitated or if you die. The GM might also decide that certain
environmental phenomena, such as a wa

## 8. Latency profiling

Each stage is timed with `time.perf_counter` after a GPU warm-up and averaged over 20 repeats on all test queries
(CUDA is synchronised before each timer stop). The LLM call is a network request to the Gemini API, measured once per
answer during section 5. Only the successful API call is timed, not the pauses between calls or any retry back-off.

The last row times the cross-encoder over **every** chunk in the corpus for one query. This is the cost of skipping the
bi-encoder stage, and is used in the answer to the scaling question.

In [9]:
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def timeit(fn, reps=20):
    fn(); sync()                                   # warm-up
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t)
    return 1000 * np.mean(ts), 1000 * np.std(ts)

qs = [q for _, q, _ in TESTS]
rows = []
m, s = timeit(lambda: [retrieve_top10(q) for q in qs], 20); rows.append(("Stage 1: bi-encoder embed query + FAISS top-10", m/len(qs), s/len(qs)))
cand_sets = {q: retrieve_top10(q) for q in qs}
m, s = timeit(lambda: [rerank(q, cand_sets[q]) for q in qs], 20); rows.append(("Stage 2: cross-encoder on 10 pairs", m/len(qs), s/len(qs)))
llm_ms = 1000 * np.nanmean([r["t_llm1"] for r in results] + [r["t_llm2"] for r in results])
rows.append(("LLM generation (Gemini API, per answer)", llm_ms, np.nan))
full = [(qs[0], c["text"]) for c in chunks]
m, s = timeit(lambda: ce.predict(full, batch_size=128), 3); rows.append((f"Cross-encoder on ALL {len(chunks)} chunks (1 query)", m, s))

lat = pd.DataFrame(rows, columns=["step", "mean_ms", "std_ms"]).round(1)
r1 = lat.mean_ms[0] + lat.mean_ms[2]
r2 = lat.mean_ms[0] + lat.mean_ms[1] + lat.mean_ms[2]
print(f"Retrieval-only part:  baseline {lat.mean_ms[0]:.1f} ms | re-rank pipeline {lat.mean_ms[0]+lat.mean_ms[1]:.1f} ms")
print(f"End-to-end incl. LLM: baseline {r1:.0f} ms | re-rank pipeline {r2:.0f} ms")
lat.to_csv(f"{OUT}/latency.csv", index=False)
lat

Retrieval-only part:  baseline 15.3 ms | re-rank pipeline 36.6 ms
End-to-end incl. LLM: baseline 964 ms | re-rank pipeline 986 ms


,step,mean_ms,std_ms
0,Stage 1: bi-encoder embed query + FAISS top-10,15.3,0.7
1,Stage 2: cross-encoder on 10 pairs,21.3,0.5
2,"LLM generation (Gemini API, per answer)",948.9,NaN
3,Cross-encoder on ALL 1504 chunks (1 query),1790.0,8.5


## 9. Scaling: extrapolating the cross-encoder cost

The cross-encoder must read a fresh `(query, chunk)` pair per chunk, so cost is linear in corpus size. Below, the measured
per-chunk cost on this corpus is extrapolated to larger collections, next to the bi-encoder search which only embeds the
query and then does a (fast) vector lookup.

In [10]:
per_chunk_ms = lat.mean_ms[3] / len(chunks)
print(f"cross-encoder cost per chunk: {per_chunk_ms:.3f} ms (GPU, batched)")
for n in (1_504, 100_000, 1_000_000, 10_000_000):
    print(f"{n:>11,} chunks -> cross-encoder over everything ~ {per_chunk_ms*n/1000:>9,.1f} s per query")
print(f"\nre-ranking only the top 10: {lat.mean_ms[1]:.1f} ms regardless of corpus size")
json.dump({"latency": lat.to_dict("records"), "per_chunk_ms": per_chunk_ms}, open(f"{OUT}/latency.json", "w"), indent=1)

cross-encoder cost per chunk: 1.190 ms (GPU, batched)
      1,504 chunks -> cross-encoder over everything ~       1.8 s per query
    100,000 chunks -> cross-encoder over everything ~     119.0 s per query
  1,000,000 chunks -> cross-encoder over everything ~   1,190.2 s per query
 10,000,000 chunks -> cross-encoder over everything ~  11,901.6 s per query

re-ranking only the top 10: 21.3 ms regardless of corpus size
